# Revisão de RNN, na prática

Neste notebook você constrói, quebra e conserta uma rede recorrente. **Não precisa anotar nada**: leia, rode e mexa no código.

Rode **uma célula de cada vez**. Algumas dão erro de propósito, e isso está avisado antes delas.

## 1. Os dados

Cada frase tem a forma `o café não|já estava <d palavras neutras> bom|ruim`, e a resposta é 1 (positiva) ou 0 (negativa).

Na função `gerar`:
- `X` guarda as frases, com cada palavra trocada pelo seu id. `y` guarda as respostas.
- `neg` é 1 se a frase tem "não". `adj` é 1 se o adjetivo é "bom".
- `adj ^ neg` é o ou exclusivo: dá 1 quando os dois valores são diferentes. Assim, "bom" sem "não" dá 1 e "ruim" com "não" também.

Treino, validação e teste têm funções diferentes: o **treino** ajusta os pesos, a **validação** acompanha o acerto sem ajustar nada, e o **teste** fica guardado para o fim. `QUATRO` são as sondas, as quatro combinações de já/não com bom/ruim.

In [ ]:
import numpy as np

VOCAB = ["o", "café", "chá", "pão", "não", "já", "estava", "bom", "ruim",
         "hoje", "aqui", "ontem", "lá", "muito", "bem"]
ID = {p: i for i, p in enumerate(VOCAB)}

def gerar(n, d, seed):
    r = np.random.default_rng(seed)
    X, y = np.zeros((n, d + 5), int), np.zeros(n)
    for i in range(n):
        neg, adj = r.integers(0, 2, size=2)
        frase = ["o", r.choice(VOCAB[1:4]), "não" if neg else "já", "estava"]
        frase += list(r.choice(VOCAB[9:], size=d)) + ["bom" if adj else "ruim"]
        X[i] = [ID[p] for p in frase]
        y[i] = adj ^ neg
    return X, y

X_tr, y_tr = gerar(3000, 8, 1)
X_val, y_val = gerar(1000, 8, 2)
X_teste, y_teste = gerar(1000, 8, 3)

NEUTRAS = ["hoje", "aqui", "ontem", "lá", "muito", "bem", "hoje", "aqui"]
QUATRO = np.array([[ID[p] for p in ["o", "café", m, "estava"] + NEUTRAS + [a]]
                   for m in ("já", "não") for a in ("bom", "ruim")])

import keras
from keras import Sequential, Input
from keras.layers import Embedding, SimpleRNN, Dense

print(X_tr.shape, QUATRO.shape)
print(" ".join(VOCAB[i] for i in X_tr[0]), "->", y_tr[0])

## 2. Construir, uma camada por vez

Aqui cada camada recebe a saída da anterior. Olhe o formato que sai de cada uma. O lote são as 4 sondas.

In [ ]:
# A entrada: 4 frases, 13 palavras em cada. Cada número é o id de uma palavra.
print(QUATRO.shape)

In [ ]:
# Embedding: troca cada id por um vetor de 8 números. O 15 é o tamanho do vocabulário.
camada_emb = Embedding(15, 8)
saida_emb = camada_emb(QUATRO)
print(saida_emb.shape)

In [ ]:
# O lote é só a quantidade de frases: com 2 frases, muda só o primeiro número.
print(camada_emb(QUATRO[:2]).shape)

In [ ]:
# SimpleRNN: lê as 13 palavras em ordem e atualiza um estado de 16 números.
# Por padrão, devolve só o último estado.
camada_rnn = SimpleRNN(16)
saida_rnn = camada_rnn(saida_emb)
print(saida_rnn.shape)

In [ ]:
# Com return_sequences=True, devolve o estado de cada uma das 13 palavras.
print(SimpleRNN(16, return_sequences=True)(saida_emb).shape)

In [ ]:
# Dense: combina os 16 números em um. A sigmoid o transforma em um valor entre 0 e 1.
# Sem treino, as probabilidades ainda não significam nada.
camada_saida = Dense(1, activation="sigmoid")
print(camada_saida(saida_rnn))

## 3. O que muda os parâmetros

Parâmetros são os números que o treino ajusta. Mude os valores à vontade e rode de novo.

In [ ]:
def montar(comprimento=13, vocab=15, dim=8, unidades=16):
    return Sequential([
        Input(shape=(comprimento,)),
        Embedding(vocab, dim),
        SimpleRNN(unidades),
        Dense(1, activation="sigmoid"),
    ])

print("modelo base      ", montar().count_params())
print("frases mais longas", montar(comprimento=30).count_params())
print("dobro de unidades ", montar(unidades=32).count_params())

## 4. Duas RNNs em sequência

A célula abaixo **dá erro de propósito**. Leia a última linha do erro.

In [ ]:
modelo = Sequential([
    Input(shape=(13,)),
    Embedding(15, 8),
    SimpleRNN(16),
    SimpleRNN(16),
    Dense(1, activation="sigmoid"),
])

Da documentação: `SimpleRNN(unidades, return_sequences=False)`. Com `return_sequences=True`, a camada devolve o estado de todos os passos.

Corrija a célula acima e confira com `modelo.summary()` o formato de saída da primeira `SimpleRNN`.

## 5. Treinar

Ler uma frase muda o estado. Treinar muda os pesos, e o estado volta a zero a cada frase.

- `compile` escolhe a **perda** (mede o quanto as respostas erraram), o **otimizador** (usa a perda para ajustar os pesos) e as **métricas** (só medem).
- `fit` roda o treino. Uma **época** é uma passada por todas as frases de treino.

A célula abaixo treina 40 épocas (cerca de 1 min) e mostra, a cada época, o acerto de validação e as probabilidades das quatro sondas.

In [ ]:
keras.utils.set_random_seed(0)
modelo = Sequential([
    Input(shape=(13,)),
    Embedding(15, 8),
    SimpleRNN(16),
    Dense(1, activation="sigmoid"),
])
modelo.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

def mostrar(epoca, logs):
    p = modelo.predict(QUATRO, verbose=0).ravel().round(2)
    print(f"época {epoca + 1:2d}  acerto {logs['val_accuracy']:.2f}  já bom {p[0]:.2f}  já ruim {p[1]:.2f}  não bom {p[2]:.2f}  não ruim {p[3]:.2f}")

h = modelo.fit(X_tr, y_tr, epochs=40, batch_size=32, validation_data=(X_val, y_val), verbose=0,
               callbacks=[keras.callbacks.LambdaCallback(on_epoch_end=mostrar)])
modelo.summary()

**Repare:** em que época o acerto sai de perto de 0,50, e o que as sondas fazem nesse momento. Queremos 1 em "já bom" e "não ruim", e 0 em "já ruim" e "não bom".

Se o acerto continuar perto de 0,50, confira dados, formatos e sondas antes de concluir qualquer coisa. Depois, mude a seed ou treine mais.

## 6. Consertar

Três bugs. Rode, descubra o que está errado, conserte na própria célula e rode de novo.

### Bug A

Esta célula **dá erro de propósito**. Leia o erro de baixo para cima: o que é o 14, e quem define o limite?

In [ ]:
modelo = Sequential([
    Input(shape=(13,)),
    Embedding(14, 8),
    SimpleRNN(16),
    Dense(1, activation="sigmoid"),
])
modelo.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
modelo.fit(X_tr, y_tr, epochs=1, batch_size=32)

### Bug B

Nada quebra aqui. Olhe o acerto, a perda e os valores de `p`. Antes de calcular a perda, o Keras corta os valores para ficarem entre quase 0 e quase 1.

In [ ]:
keras.utils.set_random_seed(0)
modelo = Sequential([
    Input(shape=(13,)),
    Embedding(15, 8),
    SimpleRNN(16),
    Dense(1),
])
modelo.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
h = modelo.fit(X_tr, y_tr, epochs=40, batch_size=32, validation_data=(X_val, y_val), verbose=0)
print("acerto e perda:", h.history["val_accuracy"][-1], h.history["val_loss"][-1])
p = modelo.predict(X_val, verbose=0).ravel()
print("menor e maior p:", p.min(), p.max())

### Bug C

O acerto fica alto, e o problema não está no modelo. Compare as sondas com o que queremos, e depois compare `gerar_b` com `gerar`.

In [ ]:
def gerar_b(n, d, seed):
    r = np.random.default_rng(seed)
    X, y = np.zeros((n, d + 5), int), np.zeros(n)
    for i in range(n):
        neg, adj = r.integers(0, 2, size=2)
        frase = ["o", r.choice(VOCAB[1:4]), "não" if neg else "já", "estava"]
        frase += list(r.choice(VOCAB[9:], size=d)) + ["bom" if adj else "ruim"]
        X[i] = [ID[p] for p in frase]
        y[i] = adj
    return X, y

X_a, y_a = gerar_b(3000, 8, 1)
X_b, y_b = gerar_b(1000, 8, 2)
keras.utils.set_random_seed(0)
modelo = Sequential([Input(shape=(13,)), Embedding(15, 8), SimpleRNN(16), Dense(1, activation="sigmoid")])
modelo.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
h = modelo.fit(X_a, y_a, epochs=10, batch_size=32, validation_data=(X_b, y_b), verbose=0)
print("acerto:", h.history["val_accuracy"][-1])
print("sondas:", modelo.predict(QUATRO, verbose=0).ravel().round(2))

# Para conferir a correção: True se os rótulos da sua função coincidem com os da função certa.
print((gerar_b(3000, 8, 1)[1] == y_tr).all())

## 7. Comparar com controle

Uma execução de cada opção não compara nada: só a seed já muda muito a época em que a rede atinge 0,95.

Complete a função abaixo, escolha **uma** mudança e rode (são 6 treinos, cerca de 6 min). Decida antes o que é "melhor" para você.

Mudanças possíveis: `unidades=8`, `unidades=32`, `dim=2`, `dim=16` ou `ini="glorot_uniform"`.

In [ ]:
def treinar(seed, unidades=16, dim=8, ini="orthogonal", teste=False):
    keras.utils.set_random_seed(seed)
    # crie o modelo usando unidades, dim e recurrent_initializer=ini
    # compile como na seção 5
    raise NotImplementedError("Crie e compile o modelo aqui, e apague esta linha.")
    h = modelo.fit(X_tr, y_tr, epochs=40, batch_size=32, validation_data=(X_val, y_val), verbose=0)
    if teste:
        return round(modelo.evaluate(X_teste, y_teste, verbose=0)[1], 2)
    # primeira época com acerto de validação >= 0,95 (None se não houve)
    primeira = next((i + 1 for i, v in enumerate(h.history["val_accuracy"]) if v >= .95), None)
    return primeira, round(h.history["val_accuracy"][-1], 2), modelo.count_params()

for seed in range(3):
    print("referência", seed, treinar(seed))
    print("mudança   ", seed, treinar(seed, unidades=32))

Só depois de decidir, avalie a opção escolhida no teste, uma única vez:

`print(treinar(0, teste=True))` para a referência, ou `print(treinar(0, unidades=32, teste=True))` para a mudança, usando a sua.

Se o teste entrasse na escolha, deixaria de ser uma medida independente.